# tabmatch — standalone demo

`tabmatch` can compare any two pandas DataFrames, regardless of whether they come from an LLM experiment. This notebook walks through common real-world scenarios:

1. Exact match (baseline)
2. Column names differ but data is the same
3. Categorical labels differ
4. Partially correct output (some columns wrong or missing)

All you need is two DataFrames that share a named index (the primary key).

In [1]:
import pandas as pd

from src.tabmatch import (
    DataComparator,
    print_comparison_report,
)

/Users/macknixon/Projects/arc/epidemiology-llm-data-tasks/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Create sample datasets

We simulate a small survey dataset with three derived variables:
- `age_group` — categorical (binned age)
- `bmi` — numeric (continuous)
- `employed` — categorical (binary employment status)

In [2]:
import numpy as np

rng = np.random.default_rng(42)
n = 200
ids = [f"P{i:04d}" for i in range(n)]

ground_truth = pd.DataFrame(
    {
        "age_group": rng.choice(["16-24", "25-34", "35-49", "50-64", "65+"], n),
        "bmi": rng.normal(26.5, 4.5, n).round(2),
        "employed": rng.choice(["Employed", "Unemployed", "Inactive"], n),
    },
    index=pd.Index(ids, name="person_id"),
)

ground_truth.head()

,age_group,bmi,employed
person_id,,,
P0000,16-24,28.30,Unemployed
P0001,50-64,22.43,Unemployed
P0002,50-64,24.80,Employed
P0003,35-49,32.35,Unemployed
P0004,35-49,24.90,Unemployed


## 2. Scenario A — perfect reproduction

The predicted output is identical to the ground truth. Every column should be a data match.

In [3]:
pred_perfect = ground_truth.copy()

comparator = DataComparator(
    categorical_data_match_threshold=0.95,
    numerical_data_match_threshold=0.0001,
)
result_perfect, _ = comparator.compare(ground_truth, pred_perfect)
print_comparison_report(result_perfect)

2026-04-29 11:42:04.609 | INFO     | src.tabmatch.data_comparator:compare:293 - Starting data comparison
2026-04-29 11:42:04.613 | INFO     | src.tabmatch.data_comparator:_check_join_completeness:122 - Join completeness: 200/200 rows (100.0%), missing=0, extra=0
2026-04-29 11:42:04.613 | INFO     | src.tabmatch.column_matcher:match_columns:139 - Matching 3 GT columns to 3 pred columns
2026-04-29 11:42:04.792 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'age_group': 1.000 (levenshtein)
2026-04-29 11:42:04.793 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'bmi': 0.045 (semantic)
2026-04-29 11:42:04.793 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'employed': 0.111 (levenshtein)
2026-04-29 11:42:04.793 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'bmi' <-> 'age_group': 0.033 (semantic)

pred_cat                             pred_n    gt_cat                                 gt_n
------------------------------------------------------------------------------------------
16-24                                    38    16-24                                    38
25-34                                    32    25-34                                    32
35-49                                    47    35-49                                    47
50-64                                    49    50-64                                    49
65+                                      34    65+                                      34
pred_cat                             pred_n    gt_cat                                 gt_n
------------------------------------------------------------------------------------------
Employed                                 58    Employed                                 58
Inactive                                 74    Inactive                                 74

───────────────────────────────────────────── DATA COMPARISON REPORT ──────────────────────────────────────────────

Primary Key Join: person_id | person_id

Task Completion Percentage:             100.0%

...

                              Column Matches                               
┏━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ GT Column ┃ Pred Column ┃ Column Match Score ┃ Data Match ┃ Method      ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ age_group │ age_group   │              1.000 │       True │ levenshtein │
│ bmi       │ bmi         │              1.000 │       True │ levenshtein │
│ employed  │ employed    │              1.000 │       True │ levenshtein │
└───────────┴─────────────┴────────────────────┴────────────┴─────────────┘

age_group ↔ age_group (categorical) 
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                  ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Exact match rate        │ 100.0% │
│ Category overlap        │  1.000 │
│ Distribution similarity │  1.000 │
│ Data Match              │   True │
└─────────────────────────┴────────┘

         bmi ↔ bmi (numeric)          
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Metric          ┃            Value ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ RMSE            │           0.0000 │
│ MAE             │           0.0000 │
│ NRMSE           │           0.0000 │
│ NMAE            │           0.0000 │
│ Correlation     │           1.0000 │
│ GT mean / std   │ 26.3569 / 4.4761 │
│ Pred mean / std │ 26.3569 / 4.4761 │
│ Data Match      │             True │
└─────────────────┴──────────────────┘

 employed ↔ employed (categorical)  
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                  ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Exact match rate        │ 100.0% │
│ Category overlap        │  1.000 │
│ Distribution similarity │  1.000 │
│ Data Match              │   True │
└─────────────────────────┴────────┘

─────────────────────────────────────────────────── END REPORT ────────────────────────────────────────────────────

## 3. Scenario B — column names differ

The predicted DataFrame uses different column names for the same variables — a common LLM behaviour. `tabmatch` uses Levenshtein and semantic similarity to recover the correct mapping without any manual configuration.

In [4]:
pred_renamed = ground_truth.rename(
    columns={
        "age_group": "age_band",  # abbreviated name
        "bmi": "body_mass_index",  # expanded name
        "employed": "employment_status",  # synonym
    }
)

result_renamed, _ = comparator.compare(ground_truth, pred_renamed)
print_comparison_report(result_renamed)

2026-04-29 11:42:04.819 | INFO     | src.tabmatch.data_comparator:compare:293 - Starting data comparison
2026-04-29 11:42:04.820 | INFO     | src.tabmatch.data_comparator:_check_join_completeness:122 - Join completeness: 200/200 rows (100.0%), missing=0, extra=0
2026-04-29 11:42:04.820 | INFO     | src.tabmatch.column_matcher:match_columns:139 - Matching 3 GT columns to 3 pred columns
2026-04-29 11:42:04.903 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'age_band': 0.632 (semantic)
2026-04-29 11:42:04.903 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'body_mass_index': 0.133 (levenshtein)
2026-04-29 11:42:04.903 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'employment_status': 0.176 (levenshtein)
2026-04-29 11:42:04.903 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'bmi' <-> 'age_band

pred_cat                             pred_n    gt_cat                                 gt_n
------------------------------------------------------------------------------------------
Employed                                 58    Employed                                 58
Inactive                                 74    Inactive                                 74
Unemployed                               68    Unemployed                               68
pred_cat                             pred_n    gt_cat                                 gt_n
------------------------------------------------------------------------------------------
16-24                                    38    16-24                                    38
25-34                                    32    25-34                                    32
35-49                                    47    35-49                                    47
50-64                                    49    50-64                                    49

───────────────────────────────────────────── DATA COMPARISON REPORT ──────────────────────────────────────────────

Primary Key Join: person_id | person_id

Task Completion Percentage:             100.0%

...

                                  Column Matches                                  
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ GT Column ┃ Pred Column       ┃ Column Match Score ┃ Data Match ┃ Method       ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ employed  │ employment_status │              0.840 │       True │ semantic     │
│ age_group │ age_band          │              0.632 │       True │ semantic     │
│ bmi       │ body_mass_index   │              1.000 │       True │ data_numeric │
└───────────┴───────────────────┴────────────────────┴────────────┴──────────────┘

    employed ↔ employment_status    
           (categorical)            
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                  ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Exact match rate        │ 100.0% │
│ Category overlap        │  1.000 │
│ Distribution similarity │  1.000 │
│ Data Match              │   True │
└─────────────────────────┴────────┘

 age_group ↔ age_band (categorical) 
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                  ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Exact match rate        │ 100.0% │
│ Category overlap        │  1.000 │
│ Distribution similarity │  1.000 │
│ Data Match              │   True │
└─────────────────────────┴────────┘

   bmi ↔ body_mass_index (numeric)    
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Metric          ┃            Value ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ RMSE            │           0.0000 │
│ MAE             │           0.0000 │
│ NRMSE           │           0.0000 │
│ NMAE            │           0.0000 │
│ Correlation     │           1.0000 │
│ GT mean / std   │ 26.3569 / 4.4761 │
│ Pred mean / std │ 26.3569 / 4.4761 │
│ Data Match      │             True │
└─────────────────┴──────────────────┘

─────────────────────────────────────────────────── END REPORT ────────────────────────────────────────────────────

## 4. Scenario C — categorical labels differ

The predicted output encodes the same categories with different labels. `tabmatch` infers the mapping from co-occurrence patterns and remaps before comparing.

In [5]:
label_map = {
    "Employed": "1 - Employed",
    "Unemployed": "2 - Unemployed",
    "Inactive": "3 - Inactive",
}
pred_relabelled = ground_truth.copy()
pred_relabelled["employed"] = pred_relabelled["employed"].map(label_map)

result_relabelled, _ = comparator.compare(ground_truth, pred_relabelled)
print_comparison_report(result_relabelled)

2026-04-29 11:42:04.924 | INFO     | src.tabmatch.data_comparator:compare:293 - Starting data comparison
2026-04-29 11:42:04.924 | INFO     | src.tabmatch.data_comparator:_check_join_completeness:122 - Join completeness: 200/200 rows (100.0%), missing=0, extra=0
2026-04-29 11:42:04.925 | INFO     | src.tabmatch.column_matcher:match_columns:139 - Matching 3 GT columns to 3 pred columns
2026-04-29 11:42:04.942 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'age_group': 1.000 (levenshtein)
2026-04-29 11:42:04.942 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'bmi': 0.045 (semantic)
2026-04-29 11:42:04.942 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'employed': 0.111 (levenshtein)
2026-04-29 11:42:04.942 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'bmi' <-> 'age_group': 0.033 (semantic)

pred_cat                             pred_n    gt_cat                                 gt_n
------------------------------------------------------------------------------------------
16-24                                    38    16-24                                    38
25-34                                    32    25-34                                    32
35-49                                    47    35-49                                    47
50-64                                    49    50-64                                    49
65+                                      34    65+                                      34
pred_cat                             pred_n    gt_cat                                 gt_n
------------------------------------------------------------------------------------------
1 - Employed                             58    Employed                                 58
2 - Unemployed                           68    Unemployed                               68

───────────────────────────────────────────── DATA COMPARISON REPORT ──────────────────────────────────────────────

Primary Key Join: person_id | person_id

Task Completion Percentage:             100.0%

...

                              Column Matches                               
┏━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ GT Column ┃ Pred Column ┃ Column Match Score ┃ Data Match ┃ Method      ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ age_group │ age_group   │              1.000 │       True │ levenshtein │
│ bmi       │ bmi         │              1.000 │       True │ levenshtein │
│ employed  │ employed    │              1.000 │       True │ levenshtein │
└───────────┴─────────────┴────────────────────┴────────────┴─────────────┘

age_group ↔ age_group (categorical) 
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                  ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Exact match rate        │ 100.0% │
│ Category overlap        │  1.000 │
│ Distribution similarity │  1.000 │
│ Data Match              │   True │
└─────────────────────────┴────────┘

         bmi ↔ bmi (numeric)          
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Metric          ┃            Value ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ RMSE            │           0.0000 │
│ MAE             │           0.0000 │
│ NRMSE           │           0.0000 │
│ NMAE            │           0.0000 │
│ Correlation     │           1.0000 │
│ GT mean / std   │ 26.3569 / 4.4761 │
│ Pred mean / std │ 26.3569 / 4.4761 │
│ Data Match      │             True │
└─────────────────┴──────────────────┘

 employed ↔ employed (categorical)  
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                  ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Exact match rate        │ 100.0% │
│ Category overlap        │  1.000 │
│ Distribution similarity │  1.000 │
│ Data Match              │   True │
└─────────────────────────┴────────┘

─────────────────────────────────────────────────── END REPORT ────────────────────────────────────────────────────

## 5. Scenario D — partial output with noise

The predicted output is missing `age_group` entirely and has a noisy version of `bmi`. This is the most realistic scenario when evaluating an LLM that only partially completed a task.

In [6]:
pred_partial = ground_truth.drop(columns=["age_group"]).copy()
pred_partial["bmi"] = (pred_partial["bmi"] + rng.normal(0, 2.0, n)).round(
    2
)  # add noise

result_partial, _ = comparator.compare(ground_truth, pred_partial)
print_comparison_report(result_partial)

2026-04-29 11:42:04.962 | INFO     | src.tabmatch.data_comparator:compare:293 - Starting data comparison
2026-04-29 11:42:04.963 | INFO     | src.tabmatch.data_comparator:_check_join_completeness:122 - Join completeness: 200/200 rows (100.0%), missing=0, extra=0
2026-04-29 11:42:04.963 | INFO     | src.tabmatch.column_matcher:match_columns:139 - Matching 3 GT columns to 2 pred columns
2026-04-29 11:42:05.173 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'bmi': 0.045 (semantic)
2026-04-29 11:42:05.173 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'age_group' <-> 'employed': 0.111 (levenshtein)
2026-04-29 11:42:05.173 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'bmi' <-> 'bmi': 1.000 (levenshtein)
2026-04-29 11:42:05.174 | DEBUG    | src.tabmatch.column_matcher:_compute_similarity_matrix:108 - Similarity 'bmi' <-> 'employed': 0.125 (levenshtein)
2026-04-2

pred_cat                             pred_n    gt_cat                                 gt_n
------------------------------------------------------------------------------------------
Employed                                 58    Employed                                 58
Inactive                                 74    Inactive                                 74
Unemployed                               68    Unemployed                               68


───────────────────────────────────────────── DATA COMPARISON REPORT ──────────────────────────────────────────────

Primary Key Join: person_id | person_id

Task Completion Percentage:             33.3%

...

                              Column Matches                               
┏━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ GT Column ┃ Pred Column ┃ Column Match Score ┃ Data Match ┃ Method      ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ bmi       │ bmi         │              1.000 │      False │ levenshtein │
│ employed  │ employed    │              1.000 │       True │ levenshtein │
│ age_group │ (unmatched) │              0.000 │          - │ -           │
└───────────┴─────────────┴────────────────────┴────────────┴─────────────┘

         bmi ↔ bmi (numeric)          
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Metric          ┃            Value ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ RMSE            │           2.0122 │
│ MAE             │           1.6019 │
│ NRMSE           │           0.0816 │
│ NMAE            │           0.0650 │
│ Correlation     │           0.9158 │
│ GT mean / std   │ 26.3569 / 4.4761 │
│ Pred mean / std │ 26.2533 / 5.0077 │
│ Data Match      │            False │
└─────────────────┴──────────────────┘

 employed ↔ employed (categorical)  
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                  ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Exact match rate        │ 100.0% │
│ Category overlap        │  1.000 │
│ Distribution similarity │  1.000 │
│ Data Match              │   True │
└─────────────────────────┴────────┘

╭───────────────────────────────────────────── Unmatched GT Columns ──────────────────────────────────────────────╮
│ age_group                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────────── END REPORT ────────────────────────────────────────────────────

## 7. Comparing two CSV files directly

For a purely file-based workflow, load both CSVs with a named index and pass them straight to `DataComparator.compare()`:

```python
gt   = pd.read_csv("ground_truth.csv", index_col="person_id")
pred = pd.read_csv("predicted.csv",    index_col="person_id")

comparator = DataComparator(
    categorical_data_match_threshold=0.95,
    numerical_data_match_threshold=0.0001,
)
result, joined = comparator.compare(gt, pred)
print_comparison_report(result)
```

The only requirement is that both files share the same index column name (the primary key).